# Validate the trained detector on AVOS (Colab)

Scores your AVOS-trained YOLOv8 checkpoint on the AVOS val split, using the
same frame-level presence/absence protocol (and statistics: Wilson 95% CI,
binomial test vs. chance) the Stage 1 hypospadias eval will use -- so the
two numbers are directly comparable once hypospadias_eval is labeled.

AVOS's val split already has expert ground truth: the YOLO bounding-box
`.txt` files from training. No new labeling happens in this notebook.

## 1. Clone the repo

In [ ]:
!git clone https://github.com/cxia0024/hypospadias-object-detection.git
%cd hypospadias-object-detection
!git checkout claude/surgical-phase-recognition-wqrp7r

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. Sanity check: run the unit tests

In [ ]:
!python -m pytest tests/ -q

## 4. Mount Google Drive

Needs your trained checkpoint and your AVOS `images/val` + `labels/val` split.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 5. Generate AVOS ground truth from its YOLO box labels

Reduces "which boxes are in this image" to "which classes are present" (the
unit Stage 1 is scored on). A missing/empty `.txt` file means no objects
were annotated in that image -- a legitimate background frame in YOLO
convention, not a labeling gap. Class order is read from the checkpoint
itself, matching the order the `.txt` files were annotated with.

In [ ]:
import sys
sys.path.insert(0, "src")
from stage1_detection.avos_labels import yolo_split_to_presence_csv
from stage1_detection.predict import get_model_classes

DRIVE_ROOT = "/content/drive/MyDrive/hypospadias"  # <-- change to your Drive folder
MODEL_PATH = f"{DRIVE_ROOT}/models/yolov8_avos_best.pt"
AVOS_IMAGES_VAL = f"{DRIVE_ROOT}/data/avos/images/val"   # <-- change to your AVOS val split
AVOS_LABELS_VAL = f"{DRIVE_ROOT}/data/avos/labels/val"   # <-- matching YOLO .txt folder

classes = get_model_classes(MODEL_PATH)
print("Classes from checkpoint:", classes)

avos_labels_csv = f"{DRIVE_ROOT}/data/avos/avos_test_labels.csv"
yolo_split_to_presence_csv(AVOS_IMAGES_VAL, AVOS_LABELS_VAL, classes, avos_labels_csv)
print(f"Wrote {avos_labels_csv}")

## 6. Write the eval config (avos_test only)

In [ ]:
import yaml

config = {
    "model_path": MODEL_PATH,
    "conf_threshold": 0.25,
    # classes are read from the checkpoint at eval time -- no need to hardcode them here.
    "datasets": {
        "avos_test": {
            "images_dir": AVOS_IMAGES_VAL,
            "labels_csv": avos_labels_csv,
            "chance_level": 0.5,
        },
    },
}

with open("configs/stage1_datasets.yaml", "w") as f:
    yaml.dump(config, f, sort_keys=False)

print(open("configs/stage1_datasets.yaml").read())

## 7. Run the evaluation

In [ ]:
!python scripts/run_stage1_eval.py --config configs/stage1_datasets.yaml --out results/avos_validation

## 8. Inspect the results

In [ ]:
import pandas as pd

pooled = pd.read_csv("results/avos_validation/pooled_metrics.csv")
per_class = pd.read_csv("results/avos_validation/per_class_metrics.csv")

print("Pooled accuracy (with Wilson 95% CI, binomial p vs. chance):")
display(pooled)

print("\nPer-class accuracy/precision/recall/F1:")
display(per_class)

## 9. Save results to Drive (optional)

In [ ]:
!cp -r results/avos_validation "$DRIVE_ROOT/results_avos_validation"
print(f"Copied to {DRIVE_ROOT}/results_avos_validation")

## Next: hypospadias zero-shot eval

Once `hypospadias_eval` frames are extracted and expert-labeled (see
`extract_frames.py` / `run_stage1_eval_colab.ipynb`), rerun
`scripts/run_stage1_eval.py` with both datasets in the config (no
`--datasets` filter) to get the AVOS-vs-hypospadias two-proportion z-test in
`pairwise_comparisons.csv`.